In [ ]:
%pip install -U langchain-openai
%pip install -U langchain-community
%pip install -U langgraph

In [12]:
import os
import re
from dotenv import load_dotenv

from langgraph.graph import StateGraph, END
from typing import TypedDict

In [13]:
# LLMs
from langchain_openai import ChatOpenAI

In [14]:
load_dotenv()

True

### Definindo o modelo llm 

In [15]:
api_key = os.getenv('OPENAI_API_KEY')
gpt_llm_model = os.getenv('GPT_LLM_MODEL')

In [18]:
model = ChatOpenAI(
  api_key=api_key,
  model=gpt_llm_model,
  temperature=0
)

In [30]:
response = model.invoke('Hello World')

In [31]:
response.content

'Hello! How can I assist you today?'

### Definindo o Agente 

In [37]:
class Agents:
  def __init__(self, system=""):
    self.system = system
    self.messages = []
    
    if self.system:
      self.messages.append({"role": "sytem", "content": self.system})
      
  def __call__(self, message):
    self.messages.append({"role": "user", "content": message})
    result = self.execute()
    self.messages.append({"role": "assistant", "content": result})
    return result
  
  def execute(self):
    # Constrói o promp com todo o histórico
    prompt = ""

    for msg in self.messages:
      prompt += f"{msg['role']}: {msg['content']}\n"
    
    response = model.invoke(prompt)
     
    return response.content

In [38]:
if __name__ == "__main__":
  agent = Agents(system="Você é um assistente útil e objetivo.")
  print(agent("Qual é a capital da franca?"))

A capital da França é Paris.


In [ ]:
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.

Ações disponíveis:
    - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
    - consultar_preço_produto: retorna o preço unitário de um produto (ex: "consultar_preço_produto: mouse gamer")

Exemplo:

Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.
""".strip()

In [39]:
class AgentState(TypedDict):
  pergunta: str
  historico: list[str]
  acao_pendente: str
  resposta_final: str

In [40]:
def consultar_estoque(item: str) -> str:
  # Simula a consulta de estoque de um item no inventário
  
  item = item.lower()
  
  estoque = {
    "monitor": 75,
    "teclado": 120,
    "mouse gamer": 80,
    "webcam": 40,
    "headset": 60,
    "impressora": 15
  }
  
  if item in estoque:
    return f"Temos {estoque[item]} {item}s em estoque."
  
  else:
    return f"Item '{item}' não encontrado no inventário."

In [41]:
def consultar_preco_produto(produto: str) -> str:
  
  produto = produto.lower()
  
  precos = {
    "monitor": 999.90,
    "teclado": 150.00,
    "mouse gamer": 99.50,
    "webcam": 120.00,
    "headset": 180.00,
    "impressora": 750.00
  }
  
  if produto in precos:
    return f"O preço de um(a) {produto} é R$ {precos[produto]:.2f}."
  else:
    return f"Produto não encontrado na lista de preços."

In [42]:
print(consultar_estoque("teclado"))

Temos 120 teclados em estoque.


In [43]:
print(consultar_preco_produto("impressora"))

O preço de um(a) impressora é R$ 750.00.
